# Custom per-bar statistics

The built-in string aggregations cover open, high, low, close, and volume.
When you want a bar feature they don't provide, any screamer functor can serve
as the reducer. This notebook shows two ways to do that: a single functor for
one custom column, and a dict of functors for several labelled columns from a
single pass.

A functor reducer follows a simple contract. It is `reset()` at every bar
boundary, fed each tick inside the bar in order, and its last output before the
bar closes becomes that bar's value. The statistic is therefore computed
causally, from the ticks in the bar and nothing else.

The examples use the expanding functors `ExpandingSkew` and `ExpandingSlope`,
which accumulate a running statistic sample by sample. That is exactly the
behaviour a bar reducer needs: accumulate within the bar, report at the close,
reset for the next one.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from screamer import ExpandingSkew, ExpandingSlope
from screamer.streams import resample

rng = np.random.default_rng(42)
n = 80

# Integer clock: each tick lands 1-2 units after the last.
timestamps = np.cumsum(rng.integers(1, 3, size=n)).astype(np.int64)

# Mid-price: a slow random walk around 100.
price = 100.0 + np.cumsum(rng.standard_normal(n) * 0.3)

# Bar width, in units of the integer clock above.
BAR = 20

## A single functor as the bar reducer

Pass a functor as `agg` and each bar reduces to that functor's closing value.
`ExpandingSkew()` reports the skewness of the price ticks inside the bar: it
accumulates the first three central moments and returns the bias-corrected
coefficient at the close. Skewness needs at least three points, so a bar with
fewer than three ticks yields `NaN`.

A single-column result unpacks as `(values, index)` and exposes `.values` and
`.index` directly.

In [ ]:
skew_bars = resample(price, timestamps, every=BAR, agg=ExpandingSkew())

values, index = skew_bars
print("bar starts:    ", index)
print("intra-bar skew:", values.round(4))

## Several statistics at once: a dict of functors

Pass a dict `{name: functor}` to run several reducers over the same bucketing in
one call. Each functor sees the same bar boundaries, and the keys become the
`.columns` of the returned `Stream`. This is the eager dict form: every value is
a functor applied to the single input stream.

Here we add `ExpandingSlope()`, which fits a least-squares line to the price
ticks against their position within the bar (0, 1, 2, ...) and returns the slope
at the close. A positive slope means price trended up inside the bar.

In [ ]:
custom = resample(price, timestamps, every=BAR,
                  agg={"skew": ExpandingSkew(), "slope": ExpandingSlope()})

print("columns:", custom.columns)
pd.DataFrame(
    custom.values,
    index=pd.Index(custom.index, name="bar_open"),
    columns=list(custom.columns),
).round(4)

In [ ]:
# Per-bar close for context, alongside the two custom features.
# All three resamples share BAR and the same clock, so they share a bar index.
close = resample(price, timestamps, every=BAR, agg="last")

fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)

axes[0].plot(close.index, close.values, marker="o", ms=5, color="steelblue")
axes[0].set_ylabel("close")
axes[0].set_title("Per-bar close, intra-bar skew, and intra-bar slope")

axes[1].plot(custom.index, custom["skew"], marker="o", ms=5, color="darkorange")
axes[1].axhline(0, color="0.6", lw=0.6)
axes[1].set_ylabel("skew")

axes[2].plot(custom.index, custom["slope"], marker="o", ms=5, color="seagreen")
axes[2].axhline(0, color="0.6", lw=0.6)
axes[2].set_ylabel("slope")
axes[2].set_xlabel("bar open index")

plt.tight_layout()